In [1]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [2]:
label_excel = r"D:\Projects\corrosions\tests\labels.xlsx"

In [3]:
df = pd.read_excel(label_excel)

In [4]:
df

,id,year,area,area_code,nomor,segment,pipe_diameter,length,segment_code,normalized_acvg_dcvg_file,normalized_cips_file,normalized_pcm_file
0,0,2025,Jakarta,jakarta-2025,1,Pipa Servis Indonesia Power,16,1.75,pipa-servis-indonesia-power-16,D:\Projects\corrosions\tests\normalize\acvg_dc...,D:\Projects\corrosions\tests\normalize\cips\ex...,D:\Projects\corrosions\tests\normalize\pcm\exc...
1,1,2025,Jakarta,jakarta-2025,2,RE Martadinata - Jl. Industri Salim Ivomas 2,16,1.70,re-martadinata-jl-industri-salim-ivomas-2-16,D:\Projects\corrosions\tests\normalize\acvg_dc...,D:\Projects\corrosions\tests\normalize\cips\ex...,D:\Projects\corrosions\tests\normalize\pcm\exc...
2,2,2025,Jakarta,jakarta-2025,3,Jl. Ps. Minggu/ SPBG - Perumahan Koperasi/Jl. ...,10,3.67,jl-ps-minggu-spbg-perumahan-koperasi-jl-g-subr...,D:\Projects\corrosions\tests\normalize\acvg_dc...,D:\Projects\corrosions\tests\normalize\cips\ex...,D:\Projects\corrosions\tests\normalize\pcm\exc...
3,3,2025,Jakarta,jakarta-2025,4,outlet MRS Pondok Ungu 1 Reducer Pipa 8'' - Te...,10,0.84,outlet-mrs-pondok-ungu-1-reducer-pipa-8-tee-va...,D:\Projects\corrosions\tests\normalize\acvg_dc...,D:\Projects\corrosions\tests\normalize\cips\ex...,D:\Projects\corrosions\tests\normalize\pcm\exc...
4,4,2025,Jakarta,jakarta-2025,5,Parang Tritis - Ancol,10,1.51,parang-tritis-ancol-10,D:\Projects\corrosions\tests\normalize\acvg_dc...,D:\Projects\corrosions\tests\normalize\cips\ex...,D:\Projects\corrosions\tests\normalize\pcm\exc...
...,...,...,...,...,...,...,...,...,...,...,...,...
90,90,2024,Cirebon,cirebon-2024,NaN,STD Garawangi - Sungai Cipetir Selatan - BV Ci...,6,5.32,std-garawangi-sungai-cipetir-selatan-bv-cilump...,D:\Projects\corrosions\tests\normalize\acvg_dc...,D:\Projects\corrosions\tests\normalize\cips\ex...,D:\Projects\corrosions\tests\normalize\pcm\exc...
91,91,2024,Cilegon,cilegon-2024,NaN,Cilegon - Merak (SV 06 Grogol - SV 07),16,4.94,cilegon-merak-sv-06-grogol-sv-07-16,NaN,D:\Projects\corrosions\tests\normalize\cips\ex...,D:\Projects\corrosions\tests\normalize\pcm\exc...
92,92,2024,Cilegon,cilegon-2024,NaN,Bojonegara - Suralaya (SV 01 - SV 04),16,16.53,bojonegara-suralaya-sv-01-sv-04-16,NaN,D:\Projects\corrosions\tests\normalize\cips\ex...,D:\Projects\corrosions\tests\normalize\pcm\exc...
93,93,2024,Cilegon,cilegon-2024,NaN,Cilegon - Anyer (SV 05 - SV 14),16,13.30,cilegon-anyer-sv-05-sv-14-16,NaN,D:\Projects\corrosions\tests\normalize\cips\ex...,D:\Projects\corrosions\tests\normalize\pcm\exc...


In [5]:
df.dtypes

id                             int64
year                           int64
area                          object
area_code                     object
nomor                         object
segment                       object
pipe_diameter                  int64
length                       float64
segment_code                  object
normalized_acvg_dcvg_file     object
normalized_cips_file          object
normalized_pcm_file           object
dtype: object

In [6]:
results = []

for index in df.index:
    row = df.iloc[index]
    protected = 0.0
    unprotected = 0.0
    medium_to_poor = 0.0
    medium_to_high = 0.0

    df_cips = pd.read_excel(row['normalized_cips_file'])
    df_pcm = pd.read_excel(row['normalized_pcm_file'])

    acvg_file = None if row['normalized_acvg_dcvg_file'] is np.nan else row['normalized_acvg_dcvg_file']
    df_acvg = pd.read_excel(acvg_file) if acvg_file is not None else None
    total_titik_anomali = len(df_acvg) if acvg_file is not None else 0

    df2 = df_cips['condition'].value_counts(normalize=True) * 100.0

    df3 = df_pcm[df_pcm['Current Loss Rate'] < 200]
    df3 = df3['Condition'].value_counts(normalize=True) * 100.0

    if 'PROTECTED' in df2.index:
        protected = df2['PROTECTED']

    if 'OVER PROTECTED' in df2.index:
        protected = df2['OVER PROTECTED'] + protected

    if 'UNPROTECTED' in df2.index:
        unprotected = df2['UNPROTECTED']

    if 'Medium to High' in df3.index:
        medium_to_high = df3['Medium to High']

    if 'Medium to Poor' in df3.index:
        medium_to_poor = df3['Medium to Poor']

    _result = {
        'Year': row['year'],
        'Area': row['area'],
        'Jalur': row['segment'],
        'Diameter (inch)': row['pipe_diameter'],
        'Panjang (km)': row['length'],
        'Protected': round(protected, 2),
        'Unprotected': round(unprotected, 2),
        'Medium to Poor': medium_to_poor,
        'Medium to High': medium_to_high,
        'Total Anomali': total_titik_anomali,
    }

    results.append(_result)


In [7]:
cips = pd.DataFrame(results)

In [8]:
cips

,Year,Area,Jalur,Diameter (inch),Panjang (km),Protected,Unprotected,Medium to Poor,Medium to High,Total Anomali
0,2025,Jakarta,Pipa Servis Indonesia Power,16,1.75,100.00,0.00,65.573770,34.426230,3
1,2025,Jakarta,RE Martadinata - Jl. Industri Salim Ivomas 2,16,1.70,100.00,0.00,49.122807,50.877193,2
2,2025,Jakarta,Jl. Ps. Minggu/ SPBG - Perumahan Koperasi/Jl. ...,10,3.67,0.44,99.56,57.894737,42.105263,2
3,2025,Jakarta,outlet MRS Pondok Ungu 1 Reducer Pipa 8'' - Te...,10,0.84,100.00,0.00,43.181818,56.818182,2
4,2025,Jakarta,Parang Tritis - Ancol,10,1.51,32.46,67.54,32.758621,67.241379,8
...,...,...,...,...,...,...,...,...,...,...
90,2024,Cirebon,STD Garawangi - Sungai Cipetir Selatan - BV Ci...,6,5.32,100.00,0.00,27.793696,72.206304,3
91,2024,Cilegon,Cilegon - Merak (SV 06 Grogol - SV 07),16,4.94,100.00,0.00,35.640138,64.359862,0
92,2024,Cilegon,Bojonegara - Suralaya (SV 01 - SV 04),16,16.53,100.00,0.00,40.022805,59.977195,0
93,2024,Cilegon,Cilegon - Anyer (SV 05 - SV 14),16,13.30,100.00,0.00,33.854167,66.145833,0


In [9]:
cips.to_excel('cips_results_segment.xlsx', index=False)

In [10]:
grouped_df = cips.groupby(['Year', 'Area'])

In [11]:
sum_df = grouped_df.sum()

In [12]:
sum_df.drop(columns='Diameter (inch)', inplace=True)

In [13]:
sum_df

Jalur  \
Year Area                                                           
2024 Bekasi     Unisma - Pd UnguUnisma - Pd UnguUnisma - Kalim...   
     Bogor                                    Jonggol - Cimanggis   
     Cilegon    Cilegon - Merak (SV 06 Grogol - SV 07)Bojonega...   
     Cirebon    STD Kedawung - STD GempolSTD Kedawung - BV Kal...   
     Jakarta    Cawang - Ahmad YaniAncol Terang - Kawasan Indu...   
     Karawang                        Cibeet - Offtake Surya Cipta   
     Tangerang  Serpong - Batu CeperBatu Ceper - BitungCikupa ...   
2025 Bekasi     Raya Kali Malang - Inspeksi Tarum Barat (Stasi...   
     Bogor      Bekasi (Narogong) - KedepDepan M/S Sukatani - ...   
     Cilegon    ST Bojonegara - SV 01 dan SV 05Stasiun Bojoneg...   
     Cirebon    STD Kedawung - BV depan MRS sektor A&BSTD Gara...   
     Jakarta    Pipa Servis Indonesia PowerRE Martadinata - Jl...   
     Karawang   Inch Curug- IndotaiseiWalahar - TexmacoTexmaco...   
     Tangerang  Depan Kawasan Langgeng Sahabat - Depan Offtake...   

                Panjang (km)  Protected  Unprotected  Medium to Poor  \
Year Area                                                              
2024 Bekasi            35.74     396.42         3.58      140.843932   
     Bogor              9.95      93.52         6.48       53.378378   
     Cilegon           37.89     300.00       100.00      134.317110   
     Cirebon           18.75     157.90       342.10      201.506678   
     Jakarta           24.50     189.83       110.17      125.364544   
     Karawang          16.92      99.95         0.05       28.439306   
     Tangerang         76.69     388.31        11.69      122.019639   
2025 Bekasi            28.00     243.28       256.72      220.196959   
     Bogor             46.30     592.32       207.68      381.660298   
     Cilegon           20.53    1000.27        99.73      503.368042   
     Cirebon           33.17     287.57       212.43      154.113878   
     Jakarta           38.82     740.17       459.83      584.147498   
     Karawang          41.00     782.08       117.92      338.245566   
     Tangerang         32.19    1632.09       667.91      872.789807   

                Medium to High  Total Anomali  
Year Area                                      
2024 Bekasi         259.156068             12  
     Bogor           46.621622              8  
     Cilegon        265.682890              0  
     Cirebon        298.493322             13  
     Jakarta        174.635456             32  
     Karawang        71.560694              3  
     Tangerang      277.980361             31  
2025 Bekasi         279.803041             58  
     Bogor          418.339702             48  
     Cilegon        596.631958              2  
     Cirebon        345.886122             32  
     Jakarta        615.852502             85  
     Karawang       561.754434             46  
     Tangerang     1427.210193            130

In [14]:
results_area = []

for year, area in sum_df.index:
    total_condition = sum_df.loc[year, area]['Protected'] + sum_df.loc[year, area]['Unprotected']
    total_protected = sum_df.loc[year, area]['Medium to Poor'] + sum_df.loc[year, area]['Medium to High']
    total_anomaly = sum_df.loc[year, area]['Total Anomali']

    _result_area = {
        'Year': year,
        'Area': area,
        'Total Panjang (km)': round(sum_df.loc[year, area]['Panjang (km)'], 2),
        'Protected': round(sum_df.loc[year, area]['Protected']/total_condition*100, 2),
        'Unprotected': round(sum_df.loc[year, area]['Unprotected']/total_condition*100, 2),
        'Medium to Poor': round(sum_df.loc[year, area]['Medium to Poor']/total_protected*100, 2),
        'Medium to High': round(sum_df.loc[year, area]['Medium to High']/total_protected*100, 2),
        'Total Anomali': total_anomaly,
    }

    results_area.append(_result_area)

In [15]:
results_area = pd.DataFrame(results_area)

In [16]:
results_area

,Year,Area,Total Panjang (km),Protected,Unprotected,Medium to Poor,Medium to High,Total Anomali
0,2024,Bekasi,35.74,99.10,0.90,35.21,64.79,12
1,2024,Bogor,9.95,93.52,6.48,53.38,46.62,8
2,2024,Cilegon,37.89,75.00,25.00,33.58,66.42,0
3,2024,Cirebon,18.75,31.58,68.42,40.30,59.70,13
4,2024,Jakarta,24.50,63.28,36.72,41.79,58.21,32
5,2024,Karawang,16.92,99.95,0.05,28.44,71.56,3
6,2024,Tangerang,76.69,97.08,2.92,30.50,69.50,31
7,2025,Bekasi,28.00,48.66,51.34,44.04,55.96,58
8,2025,Bogor,46.30,74.04,25.96,47.71,52.29,48
9,2025,Cilegon,20.53,90.93,9.07,45.76,54.24,2


In [17]:
results_area.to_excel('cips_results_area.xlsx', index=False)